# X1-S4: 통합 에이전트 데모 — Track A/B 신호 조회

## 학습 목표

이 노트북에서는 **X1 에이전트가 Track A/B 제조 AI 신호를 Tool로 조회**하는 방법을 실습합니다.

| 신호 출처 | 도구 함수 | 설명 |
|-----------|-----------|------|
| Track A-2 (AutoEncoder) | `get_anomaly_status()` | 설비 이상탐지 점수 |
| Track A-2 (LSTM) | `get_rul_prediction()` | 잔여수명(RUL) 예측 |
| Track A-2 (PdM) | `get_maintenance_schedule()` | 예지보전 정비 스케줄 |
| Track B-1 (CNN ResNet18) | `get_defect_rate()` | 품질검사 불량률 |
| Track B-2 (YOLOv8) | `get_detection_results()` | 실시간 불량 탐지 |
| 통합 | `get_manufacturing_dashboard()` | 현장 종합 대시보드 |

### 핵심 개념
- **Tool 기반 에이전트**: 에이전트가 외부 데이터를 Tool 함수로 조회
- **신호 파일 우선순위**: 실제 Track 출력 파일 → 없으면 시뮬레이션
- **ReAct 패턴**: Reason → Act(Tool 호출) → Observe(결과) → 반복
- **MCP 연동**: 같은 Tool을 Claude Code에서 MCP 서버로 사용 가능

## 1. 환경 설정 및 신호 도구 임포트

In [ ]:
import sys
import os

# tools 디렉토리를 Python 경로에 추가
# 노트북은 notebooks/ 폴더에 있으므로 한 단계 위로 이동
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"리포지토리 루트: {repo_root}")

# signal_tools 임포트
from tools.signal_tools import (
    get_anomaly_status,
    get_rul_prediction,
    get_maintenance_schedule,
    get_defect_rate,
    get_detection_results,
    get_manufacturing_dashboard,
)

print("✅ signal_tools 임포트 완료")
print("사용 가능한 Tool 함수:")
tools = [
    get_anomaly_status,
    get_rul_prediction,
    get_maintenance_schedule,
    get_defect_rate,
    get_detection_results,
    get_manufacturing_dashboard,
]
for fn in tools:
    print(f"  - {fn.__name__}(): {fn.__doc__.strip().splitlines()[0]}")

## 2. 개별 신호 조회 실습

각 Track의 신호를 직접 호출해봅니다.

> **신호 파일 탐색 순서**:
> 1. `track-a2-autoencoder-rul/outputs/signals/` (실제 학습 결과)
> 2. `track-b1-cnn-transfer/outputs/signals/`
> 3. `track-b2-vit-yolov8/outputs/signals/`
> 4. `x1-s4-agent-react/outputs/signals/` (로컬 캐시)
> 5. 파일 없으면 → **시뮬레이션 데이터 생성**

In [ ]:
# ── Track A-2: 이상탐지 신호 ────────────────────────────────────
print("=" * 60)
print("Track A-2: 이상탐지(AutoEncoder) 결과")
print("=" * 60)

anomaly = get_anomaly_status(machine_id="M001")

print(f"설비 ID    : {anomaly['machine_id']}")
print(f"상태       : {anomaly['status']}")
print(f"이상 점수  : {anomaly['anomaly_score']} (임계값: {anomaly['threshold']})")
print(f"알람 레벨  : {anomaly['alert_level']}")
print(f"데이터 출처: {anomaly['source']}")
print(f"타임스탬프 : {anomaly['timestamp'][:19]}")
print()
print(f"요약: {anomaly['summary']}")

In [ ]:
# ── Track A-2: RUL 예측 신호 ─────────────────────────────────────
print("=" * 60)
print("Track A-2: 잔여수명(RUL) 예측 결과")
print("=" * 60)

rul = get_rul_prediction(machine_id="M001")

print(f"설비 ID       : {rul['machine_id']}")
print(f"잔여수명      : {rul['rul_days']} 일")
print(f"정비 긴급도   : {rul['maintenance_urgency']}")
print(f"예측 신뢰도   : {rul['confidence']:.0%}")
print(f"데이터 출처   : {rul['source']}")
print()
print(f"요약: {rul['summary']}")

In [ ]:
# ── Track A-2: 정비 스케줄 신호 ─────────────────────────────────
print("=" * 60)
print("Track A-2: 예지보전 정비 스케줄")
print("=" * 60)

maint = get_maintenance_schedule(machine_id="M001")

print(f"설비 ID       : {maint['machine_id']}")
print(f"다음 정비일   : {maint['next_maintenance_date']}")
print(f"정비까지 남은 일수 : {maint['days_until_maintenance']}일")
print(f"우선순위      : {maint['priority']}")
print(f"권장 조치     : {maint['recommended_action']}")
print(f"예상 비용절감 : {maint['estimated_cost_saving_pct']}%")
print()
print(f"요약: {maint['summary']}")

In [ ]:
# ── Track B-1: 품질검사 신호 ─────────────────────────────────────
print("=" * 60)
print("Track B-1: 품질검사(CNN ResNet18) 결과")
print("=" * 60)

defect = get_defect_rate(line_id="LINE-A")

print(f"라인 ID       : {defect['line_id']}")
print(f"불량률        : {defect['defect_rate_pct']}")
print(f"불량 수       : {defect['defect_count']} / {defect['batch_size']} 개")
print(f"품질 등급     : {defect['quality_grade']}")
print(f"알람 여부     : {'⚠️ 발생' if defect['alert'] else '✅ 없음'}")
print(f"불량 유형 분류:")
for dtype, cnt in defect['defect_types'].items():
    print(f"  - {dtype}: {cnt}개")
print()
print(f"요약: {defect['summary']}")

In [ ]:
# ── Track B-2: 객체탐지 신호 ─────────────────────────────────────
print("=" * 60)
print("Track B-2: YOLOv8 실시간 객체탐지 결과")
print("=" * 60)

detection = get_detection_results(line_id="LINE-B")

print(f"라인 ID       : {detection['line_id']}")
print(f"탐지 불량 수  : {detection['defects_detected']} 개")
print(f"mAP@50        : {detection['map50']:.3f}")
print(f"처리 속도     : {detection['processing_fps']} FPS")
print(f"알람 여부     : {'⚠️ 발생' if detection['alert'] else '✅ 없음'}")
print(f"불량 분류:")
for cls, cnt in detection['defect_classes'].items():
    print(f"  - {cls}: {cnt}개")
print()
print(f"요약: {detection['summary']}")

## 3. 통합 대시보드 조회

`get_manufacturing_dashboard()`는 위 5개 신호를 모두 호출하고 종합 위험도를 계산합니다.

**위험도 계산 규칙**:
| 조건 | 점수 |
|------|------|
| 이상탐지 ANOMALY | +40 |
| RUL 긴급도 HIGH | +30 |
| RUL 긴급도 MEDIUM | +15 |
| 불량률 알람 | +20 |
| 탐지 알람 | +10 |

**종합 상태**: NORMAL(<30) / WARNING(30~59) / CRITICAL(60+)

In [ ]:
# 통합 대시보드 호출
dashboard = get_manufacturing_dashboard()

print("=" * 60)
print("제조 현장 통합 대시보드")
print("=" * 60)
print(f"종합 상태  : {dashboard['overall_status']}")
print(f"위험도     : {dashboard['risk_score']}/100")
print()
print("[ Track A: 설비 상태 ]")
for k, v in dashboard['track_a'].items():
    print(f"  {k:15s}: {v}")
print()
print("[ Track B: 품질 상태 ]")
for k, v in dashboard['track_b'].items():
    print(f"  {k:15s}: {v}")
print()
print("[ 권장 조치 ]")
for rec in dashboard['recommendations']:
    print(f"  {rec}")

## 4. ReAct 에이전트 + 제조 신호 도구 통합

### 시나리오
```
사용자: "M001 설비 상태 종합 점검하고 이상 있으면 작업 지시서 생성해줘"
```

에이전트는 다음 순서로 Tool을 호출합니다:
1. `get_anomaly_status("M001")` — 이상 여부 확인
2. `get_rul_prediction("M001")` — 잔여수명 확인
3. `get_maintenance_schedule("M001")` — 정비 스케줄 확인
4. 결과 종합 → 작업 지시서 생성 여부 결정

In [ ]:
from datetime import datetime

# ── SimpleReActAgent 클래스 정의 ─────────────────────────────────
class SimpleReActAgent:
    """간단한 ReAct 패턴 에이전트 구현
    
    Reason → Act(Tool 호출) → Observe(결과 확인) → 반복
    실제 LLM 없이도 Tool 호출 흐름을 학습하기 위한 데모용 구현
    """
    
    def __init__(self, tools: dict):
        # Tool 이름 → 함수 매핑 딕셔너리
        self.tools = tools
        self.history = []  # (thought, action, observation) 이력
    
    def think(self, reasoning: str):
        """Reason 단계: 다음 행동 결정 이유 기록"""
        print(f"\n🤔 [Thought] {reasoning}")
        self.history.append({"type": "thought", "content": reasoning})
    
    def act(self, tool_name: str, **kwargs) -> dict:
        """Act 단계: Tool 함수 호출"""
        if tool_name not in self.tools:
            raise ValueError(f"알 수 없는 Tool: {tool_name}")
        
        args_str = ", ".join(f"{k}={repr(v)}" for k, v in kwargs.items())
        print(f"⚡ [Action] {tool_name}({args_str})")
        
        # 실제 Tool 함수 실행
        result = self.tools[tool_name](**kwargs)
        self.history.append({"type": "action", "tool": tool_name, "args": kwargs})
        return result
    
    def observe(self, result: dict):
        """Observe 단계: Tool 실행 결과 확인"""
        summary = result.get('summary', str(result)[:100])
        print(f"👁️  [Observe] {summary}")
        self.history.append({"type": "observation", "result": summary})
        return result
    
    def finish(self, answer: str):
        """최종 답변 출력"""
        print(f"\n✅ [Final Answer]\n{answer}")
        self.history.append({"type": "answer", "content": answer})


# ── Tool 레지스트리 구성 ─────────────────────────────────────────
tool_registry = {
    "get_anomaly_status": get_anomaly_status,
    "get_rul_prediction": get_rul_prediction,
    "get_maintenance_schedule": get_maintenance_schedule,
    "get_defect_rate": get_defect_rate,
    "get_detection_results": get_detection_results,
    "get_manufacturing_dashboard": get_manufacturing_dashboard,
}

agent = SimpleReActAgent(tools=tool_registry)
print("에이전트 초기화 완료")
print(f"등록된 Tool 수: {len(tool_registry)}개")

In [ ]:
# ── 시나리오 실행 ────────────────────────────────────────────────
print("=" * 60)
print("사용자: M001 설비 상태 종합 점검하고 이상 있으면 작업 지시서 생성해줘")
print("=" * 60)

# Step 1: 이상탐지 확인
agent.think("먼저 M001 설비의 이상탐지 결과를 확인해야 한다.")
r_anomaly = agent.act("get_anomaly_status", machine_id="M001")
agent.observe(r_anomaly)

# Step 2: RUL 예측 확인
agent.think(f"이상 상태가 '{r_anomaly['status']}'임을 확인했다. 잔여수명(RUL)도 확인해보자.")
r_rul = agent.act("get_rul_prediction", machine_id="M001")
agent.observe(r_rul)

# Step 3: 정비 스케줄 확인
agent.think(f"RUL은 {r_rul['rul_days']}일로 긴급도는 '{r_rul['maintenance_urgency']}'이다. 구체적인 정비 스케줄을 확인하자.")
r_maint = agent.act("get_maintenance_schedule", machine_id="M001")
agent.observe(r_maint)

# Step 4: 작업 지시서 생성 여부 결정 및 최종 답변
needs_order = (
    r_anomaly['status'] == 'ANOMALY' or
    r_rul['maintenance_urgency'] in ('HIGH', 'MEDIUM') or
    r_maint['days_until_maintenance'] <= 14
)

agent.think(f"종합 판단: 작업 지시서 생성 필요 = {needs_order}")

if needs_order:
    # 작업 지시서 내용 생성
    work_order = f"""━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
【작업 지시서 (WO-{datetime.now().strftime('%Y%m%d-%H%M')})】
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
대상 설비  : {r_anomaly['machine_id']}
발행 일시  : {datetime.now().strftime('%Y-%m-%d %H:%M')}
우선순위   : {r_maint['priority']}

[점검 근거]
  • 이상탐지 : {r_anomaly['status']} (점수 {r_anomaly['anomaly_score']})
  • 잔여수명 : {r_rul['rul_days']}일 (긴급도 {r_rul['maintenance_urgency']})
  • 정비 예정 : {r_maint['next_maintenance_date']} ({r_maint['days_until_maintenance']}일 후)

[조치 사항]
  {r_maint['recommended_action']}

[예상 효과]
  비용절감 {r_maint['estimated_cost_saving_pct']}%
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━"""
    agent.finish(work_order)
else:
    agent.finish(f"M001 설비는 현재 정상 상태입니다. 잔여수명 {r_rul['rul_days']}일, 다음 정비일 {r_maint['next_maintenance_date']}. 작업 지시서 발행 불필요.")

## 5. 대시보드 시각화 (신호등 형태)

5개 신호의 상태를 신호등(초록/주황/빨강)으로 시각화합니다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'  # macOS 한글 폰트
matplotlib.rcParams['axes.unicode_minus'] = False

# 최신 신호 데이터 수집
dash = get_manufacturing_dashboard()
a_status  = get_anomaly_status()
r_status  = get_rul_prediction()
m_status  = get_maintenance_schedule()
d_status  = get_defect_rate()
dt_status = get_detection_results()

# ── 신호 상태 → 색상 매핑 함수 ──────────────────────────────────
def status_color(val, kind):
    """신호 값을 신호등 색상으로 변환"""
    if kind == 'anomaly':
        return '#e74c3c' if val == 'ANOMALY' else '#2ecc71'
    elif kind == 'urgency':
        return '#e74c3c' if val == 'HIGH' else ('#f39c12' if val == 'MEDIUM' else '#2ecc71')
    elif kind == 'days':
        return '#e74c3c' if val < 10 else ('#f39c12' if val < 20 else '#2ecc71')
    elif kind == 'alert':
        return '#e74c3c' if val else '#2ecc71'
    elif kind == 'overall':
        return '#e74c3c' if val == 'CRITICAL' else ('#f39c12' if val == 'WARNING' else '#2ecc71')
    return '#95a5a6'

# ── 시각화 데이터 준비 ───────────────────────────────────────────
signals = [
    {
        "name": "이상탐지\n(AutoEncoder)",
        "value": f"{a_status['status']}\n점수: {a_status['anomaly_score']}",
        "color": status_color(a_status['status'], 'anomaly'),
        "track": "Track A-2"
    },
    {
        "name": "잔여수명\n(LSTM RUL)",
        "value": f"{r_status['rul_days']:.1f}일\n긴급도: {r_status['maintenance_urgency']}",
        "color": status_color(r_status['maintenance_urgency'], 'urgency'),
        "track": "Track A-2"
    },
    {
        "name": "정비 스케줄\n(PdM)",
        "value": f"{m_status['days_until_maintenance']}일 후\n{m_status['priority']}",
        "color": status_color(m_status['days_until_maintenance'], 'days'),
        "track": "Track A-2"
    },
    {
        "name": "불량률\n(CNN ResNet18)",
        "value": f"{d_status['defect_rate_pct']}\n등급: {d_status['quality_grade']}",
        "color": status_color(d_status['alert'], 'alert'),
        "track": "Track B-1"
    },
    {
        "name": "객체탐지\n(YOLOv8)",
        "value": f"{dt_status['defects_detected']}개 탐지\nmAP: {dt_status['map50']:.3f}",
        "color": status_color(dt_status['alert'], 'alert'),
        "track": "Track B-2"
    },
]

# ── 플롯 그리기 ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 6, figsize=(16, 4))
fig.suptitle(
    f"제조 현장 신호등 대시보드  |  종합: {dash['overall_status']} (위험도 {dash['risk_score']}/100)",
    fontsize=13, fontweight='bold', y=1.02
)

# 개별 신호 원형 신호등
for ax, sig in zip(axes[:5], signals):
    circle = mpatches.Circle((0.5, 0.55), 0.28, color=sig['color'], zorder=2)
    ax.add_patch(circle)
    ax.text(0.5, 0.55, sig['value'], ha='center', va='center',
            fontsize=7.5, fontweight='bold', color='white', zorder=3,
            multialignment='center')
    ax.set_title(sig['name'], fontsize=8.5, pad=4)
    ax.text(0.5, 0.08, sig['track'], ha='center', va='bottom',
            fontsize=7, color='#7f8c8d', style='italic')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_facecolor('#f8f9fa')

# 종합 상태 패널
ax_total = axes[5]
overall_color = status_color(dash['overall_status'], 'overall')
rect = mpatches.FancyBboxPatch((0.1, 0.2), 0.8, 0.6,
    boxstyle="round,pad=0.05", linewidth=2,
    edgecolor=overall_color, facecolor=overall_color + '33')
ax_total.add_patch(rect)
ax_total.text(0.5, 0.58, dash['overall_status'],
    ha='center', va='center', fontsize=11, fontweight='bold',
    color=overall_color)
ax_total.text(0.5, 0.38, f"위험도\n{dash['risk_score']}/100",
    ha='center', va='center', fontsize=9, color='#2c3e50',
    multialignment='center')
ax_total.set_title('종합\n상태', fontsize=8.5, pad=4)
ax_total.set_xlim(0, 1)
ax_total.set_ylim(0, 1)
ax_total.axis('off')
ax_total.set_facecolor('#f8f9fa')

# 범례
legend_patches = [
    mpatches.Patch(color='#2ecc71', label='정상'),
    mpatches.Patch(color='#f39c12', label='주의'),
    mpatches.Patch(color='#e74c3c', label='위험'),
]
fig.legend(handles=legend_patches, loc='lower center',
           ncol=3, fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
plt.savefig('../outputs/signals/dashboard_snapshot.png', dpi=120,
            bbox_inches='tight', facecolor='white')
plt.show()
print("대시보드 이미지 저장: outputs/signals/dashboard_snapshot.png")

## 6. MCP 서버 연동 방법

위 Tool 함수들은 `mcp_server/manufacturing_signals_mcp.py`를 통해 **MCP 서버**로도 제공됩니다.  
Claude Code에서 다음 설정으로 같은 Tool을 자연어로 사용할 수 있습니다.

### 설치 및 실행

```bash
# 1. fastmcp 설치
pip install fastmcp

# 2. MCP 서버 실행 (테스트)
cd x1-s4-agent-react
python mcp_server/manufacturing_signals_mcp.py
```

### Claude Code MCP 등록

`~/.claude/mcp_settings.json` 에 추가:

```json
{
  "mcpServers": {
    "manufacturing-signals": {
      "command": "python",
      "args": ["/절대경로/x1-s4-agent-react/mcp_server/manufacturing_signals_mcp.py"]
    }
  }
}
```

### 사용 예시

Claude Code에서 등록 후:
```
사용자: M001 설비 지금 이상 있어?
Claude: [anomaly_status 호출] → "설비 M001: NORMAL (점수 0.234/0.75)"

사용자: 제조 현장 전체 상태 대시보드 보여줘
Claude: [manufacturing_dashboard 호출] → 종합 리포트 반환
```

### Tool vs MCP 비교

| 구분 | Tool (Python) | MCP 서버 |
|------|---------------|----------|
| 사용 방법 | `get_anomaly_status()` 직접 호출 | Claude가 자연어로 호출 |
| 적합한 환경 | Jupyter 노트북, Python 스크립트 | Claude Code, LLM 에이전트 |
| 인증/권한 | Python 환경 권한 | MCP 서버 레벨 제어 가능 |
| 내부 구현 | 동일한 `signal_tools.py` 함수 사용 | 동일한 `signal_tools.py` 함수 사용 |

In [ ]:
# ── 마무리: 전체 Tool 요약 테스트 ───────────────────────────────
print("=" * 60)
print("전체 Tool 요약")
print("=" * 60)

results = [
    get_anomaly_status(),
    get_rul_prediction(),
    get_maintenance_schedule(),
    get_defect_rate(),
    get_detection_results(),
]

for i, r in enumerate(results, 1):
    print(f"{i}. {r['summary']}")

print()
final = get_manufacturing_dashboard()
print(f"📊 {final['summary']}")
print()
print("권장 조치:")
for rec in final['recommendations']:
    print(f"  {rec}")